In [ ]:
import os
from pathlib import Path
# imports
import logging
import sys

import nibabel as nib
import numpy as np
import pandas as pd
import torch
from torch.utils.tensorboard import SummaryWriter

import monai
from monai.apps import download_and_extract
from monai.config import print_config
from monai.data import DataLoader, ImageDataset
from monai.transforms import (
    Transform,
    Spacing,
    EnsureChannelFirst,
    Compose,
    RandRotate90,
    Resize,
    ScaleIntensity,
    Lambda,
    CropForeground,
    ScaleIntensityRangePercentiles,
    SpatialPad
    
)

pin_memory = torch.cuda.is_available()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
print_config()

# Configure local data and run paths with environment variables.
DATA_ROOT = Path(os.environ.get("MRI_DATA_ROOT", "data"))
METADATA_CSV = Path(
    os.environ.get("MRI_METADATA_CSV", DATA_ROOT / "filtered_df_for_share.csv")
)
MODEL_DIR = Path(os.environ.get("MRI_MODEL_DIR", "."))
OUTPUT_DIR = Path(os.environ.get("MRI_OUTPUT_DIR", "outputs"))
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# get region labels from csv

label_column = 'classification.region_label'
# label_column = 'classification.sequence_label'
# label_column = 'classification.fat_suppression_label' 
# label_column = 'classification.contrast_combined'
split_column = 'split.region'
# split_column = 'split.sequence' 
# split_column = 'split.fat_sup' 
# split_column = 'split.contrast'
flywheel_info = pd.read_csv(METADATA_CSV, low_memory=False)

root_dir = str(DATA_ROOT)

# add root dir to image file names
flywheel_info['flywheel_info.nifti_name'] = root_dir + '/' + flywheel_info['flywheel_info.nifti_name']
image_files = flywheel_info['flywheel_info.nifti_name']

# map image_labels to numerical, generate the map via pandas!
mapper = {val: i for i, val in enumerate(flywheel_info["classification.contrast_combined"].unique())}
flywheel_info['classification.contrast_combined'] = flywheel_info['classification.contrast_combined'].map(mapper)
image_labels = flywheel_info['classification.contrast_combined']

# indices to use when creating the train/val split loaders
label_groups = {label: group.index for label, group in flywheel_info.groupby('split.region')}

print(f"train: {len(image_files[label_groups['train']])}, val: {len(image_files[label_groups['val']])}, test:{len(image_files[label_groups['test']])}, total:{len(image_files)}")
print(f"category mapper: {mapper}")


In [ ]:
# Define transforms
def squeeze_to_one_channel(x):
    # if C==1, do nothing
    if x.shape[0] == 1:
        return x
    # if C>1 (e.g., DWI), keep channel 0
    return x[0:1, ...]  


class MoveToLastDim(Transform):
    def __init__(self):
        super().__init__()
    
    def __call__(self, data):
        # (200, 300, 400) --> (300, 400, 200)
        shape = data.shape
        min_dim_idx = int(np.argmin(shape))
        new_axes = [i for i in range(len(shape)) if i != min_dim_idx]
        new_axes.append(min_dim_idx)
        
        if hasattr(data, 'permute'):
            return data.permute(*new_axes)
        else:
            return data.transpose(new_axes)


class SquarePad(Transform):
    def __init__(self):
        super().__init__()
    
    def __call__(self, data):
        # (200, 300, 400) --> (300, 300, 400)
        pad_size = max(data.shape[0], data.shape[1])
        return SpatialPad((pad_size, pad_size, -1))(data)  # call it on data
        
train_transforms = Compose([EnsureChannelFirst(),
                            # Lambda(squeeze_to_one_channel),
                            Spacing(pixdim=[1.0, 1.0, 1.0]),
                            CropForeground(allow_smaller=True),
                            ScaleIntensityRangePercentiles(lower=1, upper=99, b_min=0.0, b_max=1.0, clip=True),
                            # MoveToLastDim(),
                            SquarePad(),
                            Resize((128,128,128))])

val_transforms = Compose([EnsureChannelFirst(), 
                          # Lambda(squeeze_to_one_channel),
                          Spacing(pixdim=[1.0, 1.0, 1.0]),
                          CropForeground(allow_smaller=True),
                          ScaleIntensityRangePercentiles(lower=1, upper=99, b_min=0.0, b_max=1.0, clip=True),
                          # MoveToLastDim(),
                          SquarePad(),
                          Resize((128,128,128))])

# Define nifti dataset, data loader
check_ds = ImageDataset(image_files=image_files.tolist(), labels=image_labels.tolist(), transform=train_transforms)
check_loader = DataLoader(check_ds, batch_size=4, num_workers=0, pin_memory=pin_memory)

# im, label = monai.utils.misc.first(check_loader)
# print(type(im), im.shape, label, len(label))

# create a training data loader
train_ds = ImageDataset(image_files=image_files[label_groups['train']].tolist(), labels=image_labels[label_groups['train']].tolist(), transform=train_transforms)
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0, pin_memory=pin_memory)

# create a validation data loader
val_ds = ImageDataset(image_files=image_files[label_groups['val']].tolist(), labels=image_labels[label_groups['val']].tolist(), transform=val_transforms)
val_loader = DataLoader(val_ds, batch_size=1, num_workers=0, pin_memory=pin_memory)

# create a test data loader
test_ds = ImageDataset(image_files=image_files[label_groups['test']].tolist(), labels=image_labels[label_groups['test']].tolist(), transform=val_transforms)
test_loader = DataLoader(test_ds, batch_size=1, num_workers=8, pin_memory=pin_memory)

In [ ]:
import itertools
import pandas as pd

cols = ['split.region', 'split.sequence', 'split.fat_sup', 'split.contrast']
values = ['train', 'val', 'test']

for c1, c2 in itertools.combinations(cols, 2):
    ct = pd.crosstab(flywheel_info[c1], flywheel_info[c2]).reindex(index=values, columns=values, fill_value=0)
    agree = ct.to_numpy().trace() / ct.to_numpy().sum()
    print(f"{c1} (rows) vs {c2} (cols) -- {agree:.1%} same assignment")
    print(ct, "\n")

In [ ]:
###### reload the best model
import os

model = monai.networks.nets.DenseNet121(spatial_dims=3, in_channels=1, out_channels=6).to(device)
model.load_state_dict(torch.load(MODEL_DIR / "best_metric_model_classification3d_array.pth"))
model.eval()

# recreate the test data loader
# should exist from the original build, but copy here just in case
def squeeze_to_one_channel(x):
    # if C==1, do nothing
    if x.shape[0] == 1:
        return x
    # if C>1 (e.g., DWI), keep channel 0
    return x[0:1, ...]  

val_transforms = Compose([ScaleIntensity(), 
                          EnsureChannelFirst(), 
                          Lambda(squeeze_to_one_channel),
                          Resize((96, 96, 96))])
val_ds = ImageDataset(image_files=image_files[label_groups['val']].tolist(), labels=image_labels[label_groups['val']].tolist(), transform=val_transforms)
val_loader = DataLoader(val_ds, batch_size=2, num_workers=8, pin_memory=pin_memory)
val_labels_true=image_labels[label_groups['val']].tolist()

test_ds = ImageDataset(image_files=image_files[label_groups['test']].tolist(), labels=image_labels[label_groups['test']].tolist(), transform=val_transforms)
test_loader = DataLoader(test_ds, batch_size=2, num_workers=8, pin_memory=pin_memory)
test_labels_true=image_labels[label_groups['test']].tolist()

In [ ]:
num_correct = 0.0
metric_count = 0
for val_data in val_loader:
    val_images, val_labels = val_data[0].to(device), val_data[1].to(device)
    with torch.no_grad():
        val_outputs = model(val_images)
        # value = torch.eq(val_outputs.argmax(dim=1), val_labels.argmax(dim=1))
        # 2025-12 csk add Dvij suggestion
        pred = val_outputs.argmax(dim=1)
        value = pred == val_labels
        metric_count += len(value)
        num_correct += value.sum().item()

metric = num_correct / metric_count
print("metric", metric)

In [ ]:
import torch.nn.functional as F
import time

model.eval()
y_pred, y_probs, y_logits = [], [], []

t1 = time.time()
with torch.no_grad():
    for test_data in test_loader:
        test_images = test_data[0].to(device)

        logits = model(test_images)             # shape [B, C]
        probs = F.softmax(logits, dim=1)        # shape [B, C], rows sum to 1

        y_logits.append(logits.cpu().numpy())
        y_probs.append(probs.cpu().numpy())
        y_pred.append(probs.argmax(dim=1).cpu().numpy())   # shape [B]

t2 = time.time()
print(f"test time: {t2 - t1:.1f}s")

all_logits = np.concatenate(y_logits)   # (N, C)
all_probs = np.concatenate(y_probs)     # (N, C)
all_preds = np.concatenate(y_pred)      # (N,)

In [ ]:

print(label_column)

In [ ]:
# make a new dataframe with the saved columns, and save it to csv
label_column = 'classification.region_label'
test_results = flywheel_info.loc[label_groups['test'], ['flywheel_info.subject_label', 'flywheel_info.session_label', 'flywheel_info.acq_label', label_column]].copy()
test_results["pred_label"] = all_preds
logit_cols = [f"logit_{j}" for j in range(6)]
prob_cols = [f"prob_{j}" for j in range(6)]
test_results[logit_cols] = all_logits
test_results[prob_cols] = all_probs

test_results.to_csv(OUTPUT_DIR / f'{label_column}_predictions.csv', index=False)